# 01 — Setup, Annotations Download, and Video Acquisition

This notebook prepares the AVUT-Human dataset for evaluation.

**What it does:**
1. Installs dependencies (transformers, accelerate, qwen-omni-utils, openai-whisper, yt-dlp, ffmpeg-python).
2. Downloads the AVUT annotation JSONs from HuggingFace (`tsinghua-ee/AVUTBenchmark`).
3. Inspects the JSON schema and prints a sample entry — **STOP and verify field names match `src/data_utils.py`**.
4. Downloads videos from YouTube via `yt-dlp` and logs success rate per task.

**Runtime:** ~2–4 hours (mostly YouTube downloads). Resume-safe: rerun to skip already-downloaded videos.

**GPU:** Not required for this notebook; H100 is wasted here. Use a CPU runtime if available.


## 0. Environment

If running outside Colab, mount the repo manually. In Colab, clone the repo first:

In [ ]:
# Colab: clone repo (skip if running locally)
import os, sys
REPO = '/content/omnimodel-research'
if not os.path.exists(REPO):
    # Replace with your fork/branch as needed
    !git clone https://github.com/<you>/omnimodel-research.git /content/omnimodel-research
%cd $REPO
sys.path.insert(0, REPO)


In [ ]:
# Install dependencies. ffmpeg is preinstalled on Colab.
!pip install -q transformers>=4.45 accelerate sentencepiece
!pip install -q yt-dlp openai-whisper
!pip install -q qwen-omni-utils
!pip install -q huggingface_hub


## 1. Download AVUT annotations

In [ ]:
from huggingface_hub import hf_hub_download
import json, os

os.makedirs('data/annotations', exist_ok=True)

for fname in ['AV_Human_data.json', 'AV_Gemini_data.json']:
    try:
        path = hf_hub_download(
            repo_id='tsinghua-ee/AVUTBenchmark',
            filename=fname,
            repo_type='dataset',
            local_dir='data/annotations',
        )
        print(f'OK: {fname} → {path}')
    except Exception as e:
        print(f'FAILED: {fname}: {e}')


## 2. Inspect JSON schema (CRITICAL — verify before proceeding)

In [ ]:
with open('data/annotations/AV_Human_data.json') as f:
    raw = json.load(f)

print('Type at top level:', type(raw).__name__)
entries = raw if isinstance(raw, list) else list(raw.values())
print(f'Number of entries: {len(entries)}')
print()
print('First entry — full schema:')
print(json.dumps(entries[0], indent=2)[:2000])
print()
print('All keys present in first 50 entries:')
keys = set()
for e in entries[:50]:
    if isinstance(e, dict):
        keys.update(e.keys())
print(sorted(keys))


In [ ]:
# Task type distribution (uses src/data_utils.py with its tolerant field-name handling)
from src.data_utils import load_annotations, filter_to_target_tasks, task_distribution

normalized = load_annotations('data/annotations/AV_Human_data.json')
print(f'Normalized entries (with valid video_id, question, options, answer): {len(normalized)}')
print()
print('Task distribution (all tasks):')
print(json.dumps(task_distribution(normalized), indent=2))
print()
print('Task distribution (MCQ tasks we evaluate):')
mcq = filter_to_target_tasks(normalized)
print(json.dumps(task_distribution(mcq), indent=2))
print(f'Total MCQ entries: {len(mcq)}')


## 3. Download videos via yt-dlp

We download up to 720p to keep storage reasonable. Since we'll later sample 120 entries, we can either:
- (a) Download videos for all ~700 AV-Human entries (~50–100 GB), then sample.
- (b) Sample first, then download only the 120-sample subset (~10 GB, much faster).

We use approach **(b)** for speed. If the sampled set has too many download failures, increase the sample size and resample.


In [ ]:
from src.data_utils import balanced_subsample
import subprocess

# AVUT has 692 unique videos but 1734 QA pairs (each video has ~2.5 QAs across
# different task types). Multiple QAs can share the same downloaded video.
# We sample BY QUESTION (qa_id) but DEDUPE downloads by video_id.

# Sample more than we need so we have headroom for failed YouTube downloads.
N_PER_TASK_TO_DOWNLOAD = 30   # we'll keep 20/task after filtering for download success
SEED = 42

candidate_pool = balanced_subsample(mcq, n_per_task=N_PER_TASK_TO_DOWNLOAD, seed=SEED)
unique_videos = list({s['video_id']: s for s in candidate_pool}.values())
print(f'Candidate QA pairs: {len(candidate_pool)} ({N_PER_TASK_TO_DOWNLOAD}/task × 6 tasks)')
print(f'Unique videos to download: {len(unique_videos)}')

# Save the candidate set so we can re-run downstream notebooks deterministically
os.makedirs('data', exist_ok=True)
with open('data/candidate_samples.json', 'w') as f:
    json.dump(candidate_pool, f, indent=2)
print('Saved candidate QA list to data/candidate_samples.json')


In [ ]:
# Download videos. Each AVUT entry already carries the YouTube URL inline
# (`url` field), so no separate mapping file is needed.

def youtube_url_for(sample):
    return sample['url']  # AVUT annotations include the URL directly


In [ ]:
os.makedirs('data/videos', exist_ok=True)
download_log = []

for i, sample in enumerate(unique_videos):
    vid = sample['video_id']
    out_path = f'data/videos/{vid}.mp4'
    if os.path.exists(out_path):
        download_log.append({'video_id': vid, 'status': 'cached'})
        continue
    url = youtube_url_for(sample)
    try:
        result = subprocess.run(
            [
                'yt-dlp',
                '-f', 'bestvideo[height<=720]+bestaudio/best[height<=720]',
                '--merge-output-format', 'mp4',
                '-o', out_path,
                '--no-playlist',
                '--socket-timeout', '30',
                '--quiet',
                url,
            ],
            timeout=180,
            check=False,
            capture_output=True,
        )
        ok = os.path.exists(out_path)
        download_log.append({'video_id': vid, 'status': 'ok' if ok else 'failed',
                             'returncode': result.returncode})
    except subprocess.TimeoutExpired:
        download_log.append({'video_id': vid, 'status': 'timeout'})
    if (i + 1) % 10 == 0:
        succ = sum(1 for d in download_log if d['status'] in ('ok', 'cached'))
        print(f'  [{i+1}/{len(unique_videos)}] success={succ}/{i+1}')

from collections import Counter
status_counts = Counter(d['status'] for d in download_log)
print()
print('Per-video download status:', dict(status_counts))
n_avail = sum(1 for d in download_log if d['status'] in ('ok', 'cached'))
print(f'Videos available on disk: {n_avail}/{len(download_log)}')

# Now compute QA-level availability (a QA is available iff its video is)
avail_vids = {d['video_id'] for d in download_log if d['status'] in ('ok', 'cached')}
qa_avail = [s for s in candidate_pool if s['video_id'] in avail_vids]
print(f'QA pairs available: {len(qa_avail)}/{len(candidate_pool)}')

per_task = Counter(s['task_type'] for s in qa_avail)
print('Per-task QA availability:')
for t, n in per_task.most_common():
    print(f'  {t}: {n}')

with open('data/download_log.json', 'w') as f:
    json.dump(download_log, f, indent=2)


## 4. Lock the final balanced eval set

Filter the candidate pool to videos actually on disk, then sub-sample to exactly 20 per task. This is the canonical sample list every downstream notebook reads.

In [ ]:
from src.data_utils import filter_to_available_videos, balanced_subsample, task_distribution_by_code

available = filter_to_available_videos(candidate_pool, video_dir='data/videos')
print(f'QA pairs with video on disk: {len(available)} / {len(candidate_pool)}')
print('Per-task available:', task_distribution_by_code(available))

N_PER_TASK_FINAL = 20
final_set = balanced_subsample(available, n_per_task=N_PER_TASK_FINAL, seed=42)
print()
print(f'Final eval set: {len(final_set)} QA pairs ({N_PER_TASK_FINAL}/task)')
print('Per-task final:', task_distribution_by_code(final_set))

with open('data/eval_samples.json', 'w') as f:
    json.dump(final_set, f, indent=2)
print('Saved final eval sample list to data/eval_samples.json')


**Done.** Proceed to `02_preprocess.ipynb`.